In [ ]:
from huggingface_hub import login
login()


In [ ]:
from datasets import load_dataset

dataset = load_dataset("wmcnicho/StudyChat", streaming=True)

for example in dataset["train"]:
    print(example)
    break


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.47k [00:00<?, ?B/s]

{'prompt': 'cross validation tests it on multiple sets, so does it randomly select data on my dataset? concise', 'response': 'Yes, cross-validation randomly splits your dataset into multiple subsets (or "folds"). It tests the model on one subset while training it on the others, ensuring that the model\'s performance is evaluated on different sections of the data multiple times. This helps provide a more reliable assessment of the model\'s ability to generalize to new data.', 'topic': 'a4', 'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'The R-squared score represents the proportion of the variance in the dependent variable (Size nm³) that is predictable from the independent variables (Temperature and Mols KCl). An R-squared score of 1 indicates perfect prediction, while a score of 0 indicates that the model does not explain any of the variability in the target variable\nexplain this', 'role': 'user'}, {'content': 'Certainly! The R-squared score,

In [ ]:
from itertools import islice
import pandas as pd
import numpy as np
import re
import re
import json
import random
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple


In [ ]:


# 1) stream the dataset
ds = load_dataset("wmcnicho/StudyChat", streaming=True)
train = ds["train"]

# 2) take a small sample (e.g., 2000 chats)
sample_n = 2000
sample = list(islice(train, sample_n))

# 3) use this df to test functions
df = pd.DataFrame(sample)

print(df.columns)
print(df.iloc[0]["messages"][:4])


Index(['prompt', 'response', 'topic', 'messages', 'timestamp', 'chatId',
       'userId', 'interactionCount', 'chatTitle', 'chatStartTime',
       'chatTotalInteractionCount', 'llm_label', 'semester'],
      dtype='object')
[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'The R-squared score represents the proportion of the variance in the dependent variable (Size nm³) that is predictable from the independent variables (Temperature and Mols KCl). An R-squared score of 1 indicates perfect prediction, while a score of 0 indicates that the model does not explain any of the variability in the target variable\nexplain this', 'role': 'user'}, {'content': 'Certainly! The R-squared score, often denoted as \\( R^2 \\), is a statistical measure used in the context of regression models. It provides insight into how well the independent variables (predictors) explain the variability or variation of the dependent variable (outcome). Here’s a breakdown of the key concepts

In [ ]:
print(df.size) #df has 26000 lines

26000


In [ ]:
print(df["chatId"].isna().mean())
print(df["chatId"].nunique(),len(df))

0.0
311 2000


In [ ]:
"""Data Preparation"""
import numpy as np
import re

KEEP_COLS = [
    "prompt", "response", "topic", "messages", "chatId", "userId", "interactionCount",
    "chatTitle", "chatTotalInteractionCount", "llm_label"
]

def word_count(text: str) -> int:
    # counts "words" as sequences of letters/numbers, robust to punctuation
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\b\w+\b", text.strip()))

def extract_user_messages(messages):
    """messages: list[{'content':..., 'role':...}, ...]"""
    if not isinstance(messages, list):
      return []
    out = []
    for m in messages:
      if isinstance(m, dict) and m.get("role") == "user":
        txt = m.get("content", "")
        if isinstance(txt, str) and txt.strip():
          out.append(txt.strip())
    return out


def restructure_for_label(
    df,
    prompt_col="prompt",
    response_col="response",
    chat_id_col="chatId",
    seg_idx_col="interactionCount",
    chat_total_col="chatTotalInteractionCount",
):
    """
    Use only prompt/response for labeling.
    Ensure rows are ordered within each chatId by interactionCount ascending (0,1,2,...).
    """

    d = df.copy()

    # remove emplty/invalid chats
    if chat_total_col in d.columns:
      d = d[d[chat_total_col]>0].copy()

    # 0) make sure interactionCount is numeric for correct sorting
    d[seg_idx_col] = pd.to_numeric(d[seg_idx_col], errors="coerce")

    # 1) generate unique conversation_id (chatId_interactionCount)
    d["conversation_id"] = d[chat_id_col].astype(str) + "_" + d[seg_idx_col].astype("Int64").astype(str)

    # 2) simple features from prompt (you can also add from response if you want)
    d["prompt_word_len"] = d[prompt_col].fillna("").apply(lambda s: len(str(s).split()))
    d["n_qmarks"] = d[prompt_col].fillna("").apply(lambda s: str(s).count("?"))
    d["n_#marks"] = d[prompt_col].fillna("").apply(lambda s: str(s).count("#"))
    d["prompt_char_len"] = d[prompt_col].fillna("").apply(lambda s: len(str(s)))

    # 3) keep columns
    keep = [
        "conversation_id",
        chat_id_col,
        seg_idx_col,
        chat_total_col if chat_total_col in d.columns else None,
        prompt_col,
        response_col,
        "prompt_word_len",
        "prompt_char_len",
        "n_qmarks",
        "n_#marks",
    ]
    keep = [c for c in keep if c is not None]

    # keep llm_label if exists
    if "llm_label" in d.columns:
        keep.append("llm_label")

    # 4) sort: first by chatId, then by interactionCount ascending
    out = (
        d[keep]
        .sort_values(by=[chat_id_col, seg_idx_col], ascending=[True, True])
        .reset_index(drop=True)
       )

    return out

def sample_full_chats(df, chat_id_col="chatId", n_chats=200, seed=42):
    rng = np.random.default_rng(seed)
    chat_ids = df[chat_id_col].dropna().unique()
    picked = rng.choice(chat_ids, size=min(n_chats, len(chat_ids)), replace=False)
    out = df[df[chat_id_col].isin(picked)].copy()
    out = out.sort_values([chat_id_col, "interactionCount"]).reset_index(drop=True)
    return out


In [ ]:
# if already have a pandas df that includes the columns you listed:
turn_df = restructure_for_label(df)

turn_df.head()

,conversation_id,chatId,interactionCount,chatTotalInteractionCount,prompt,response,prompt_word_len,prompt_char_len,n_qmarks,n_#marks,llm_label
0,006b74d1-d88c-43ce-8e22-e27ac270a79d_0,006b74d1-d88c-43ce-8e22-e27ac270a79d,0,8,How to get all of the unique characters from a...,To get all of the unique characters from a str...,11,53,0,0,{'label': 'conceptual_questions>Programming La...
1,006b74d1-d88c-43ce-8e22-e27ac270a79d_1,006b74d1-d88c-43ce-8e22-e27ac270a79d,1,8,"is this correct? char_to_idx = {char for idx, ...",The code snippet you provided is actually inco...,11,71,1,0,"{'label': 'verification>Verify Code', 'justifi..."
2,006b74d1-d88c-43ce-8e22-e27ac270a79d_2,006b74d1-d88c-43ce-8e22-e27ac270a79d,2,8,"char_to_idx = {char:idx for idx, char in enume...",The code snippet you provided contains an erro...,16,117,0,0,"{'label': 'provide_context>Code', 'justificati..."
3,006b74d1-d88c-43ce-8e22-e27ac270a79d_3,006b74d1-d88c-43ce-8e22-e27ac270a79d,3,8,how does pytorch test train split,"In PyTorch, the typical way to handle train-te...",6,33,0,0,{'label': 'conceptual_questions>Programming To...
4,006b74d1-d88c-43ce-8e22-e27ac270a79d_4,006b74d1-d88c-43ce-8e22-e27ac270a79d,4,8,what does this mean: # This is Cell #9\n\ntrai...,The comment `# This is Cell #9` is likely a no...,46,354,0,3,{'label': 'contextual_questions>Assignment Cla...


In [ ]:
print(turn_df.shape)

(1921, 11)


In [ ]:
df_sample = sample_full_chats(turn_df, chat_id_col="chatId", n_chats=200, seed=42)

df_sample.head()

,conversation_id,chatId,interactionCount,chatTotalInteractionCount,prompt,response,prompt_word_len,prompt_char_len,n_qmarks,n_#marks,llm_label
0,00f9ebab-7e4c-471a-8623-523812e2bd4d_0,00f9ebab-7e4c-471a-8623-523812e2bd4d,0,20,from collections import deque\nimport heapq\ni...,Your implementation for the `Autocomplete` cla...,358,3871,0,15,"{'label': 'provide_context>Code', 'justificati..."
1,00f9ebab-7e4c-471a-8623-523812e2bd4d_1,00f9ebab-7e4c-471a-8623-523812e2bd4d,1,20,justwhy@vl965-172-31-13-247 assignment-2-searc...,The error message you're encountering indicate...,39,676,0,0,"{'label': 'provide_context>Error Message', 'ju..."
2,00f9ebab-7e4c-471a-8623-523812e2bd4d_2,00f9ebab-7e4c-471a-8623-523812e2bd4d,2,20,how do I run it such that I can see a demo of ...,To run a demonstration of the three search fun...,17,69,1,0,{'label': 'conceptual_questions>Programming To...
3,00f9ebab-7e4c-471a-8623-523812e2bd4d_3,00f9ebab-7e4c-471a-8623-523812e2bd4d,3,20,this is my current main.py:\nfrom autocomplete...,Looking at your current `main.py` and `utiliti...,112,1314,0,0,{'label': 'contextual_questions>Assignment Cla...
4,00f9ebab-7e4c-471a-8623-523812e2bd4d_4,00f9ebab-7e4c-471a-8623-523812e2bd4d,4,20,im not allowed to modify utilities.py,If you are not allowed to modify `utilities.py...,6,37,0,0,{'label': 'contextual_questions>Assignment Cla...


In [ ]:
print(df_sample.shape)

(1747, 11)


In [ ]:
print(df_sample['chatId'].nunique(), len(df_sample))

200 1747


In [ ]:
# Save the DataFrame to an Excel file
df_sample.to_excel("df_sample.xlsx", index=False)

print("DataFrame saved to df_sample.xlsx")

DataFrame saved to df_sample.xlsx


In [ ]:
# ============================================================
# Epistemic Aims + Epistemic Processes Coding Pipeline (Starter)
# Methods: Manual coding / Regex rules / LLM classifier
# ============================================================


import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

# Optional: for agreement stats
from sklearn.metrics import cohen_kappa_score, classification_report

# ----------------------------
# 0) CONFIG: column names
# ----------------------------
TEXT_COL = "prompt"
ID_COL = "conversation_id"

KEEP_META_COLS = ["interactionCount", "topic", "llm_label", "response"]

def ensure_id_column(df: pd.DataFrame, id_col: str = ID_COL) -> pd.DataFrame:
    if id_col not in df.columns:
        df = df.copy()
        df[id_col] = range(len(df))
    return df


# ============================================================
# 1) Sampling for manual coding (balanced / random)
# ============================================================

def make_annotation_sheet(df: pd.DataFrame, text_col: str = TEXT_COL) -> pd.DataFrame:


    """
    Create a manual annotation sheet with blank columns for your 2-level codebook:
      - Aims: relevance (on/off), conceptual, procedural
      - Processes: outsourcing, explanation, verification, prompt_improvement, regulation
    """
    df = df.copy()

    # LEVEL 1: Epistemic aims
    df["aim_relevance"] = ""     # on-topic / off-topic
    df["aim_understanding"] = ""    # 0/1

    # LEVEL 2: Epistemic processes
    df["proc_outsourcing"] = ""       # 0/1
    df["proc_explanation"] = ""       # 0/1
    df["proc_verification"] = ""      # 0/1
    df["proc_prompt_improve"] = ""    # 0/1
    df["proc_regulation"] = ""        # 0/1

    df["annotator_notes"] = ""

    cols = [ID_COL]
    for c in KEEP_META_COLS:
        if c in df.columns:
            cols.append(c)

    cols += [text_col,
             "aim_relevance", "aim_understanding",
             "proc_outsourcing", "proc_explanation", "proc_verification",
             "proc_prompt_improve", "proc_regulation",
             "annotator_notes"]

    # Keep only existing cols
    cols = [c for c in cols if c in df.columns]
    return df[cols].copy()

# export the annotation sheet
anno_sheet = make_annotation_sheet(df_sample)
anno_sheet.to_excel("anno_sheet.xlsx", index=False)

print("DataFrame saved to anno_sheet.xlsx")

DataFrame saved to anno_sheet.xlsx


# LLM_Labeling Few-Shot Prompting

In [ ]:
!pip install openai
!pip install langchain
!pip install langchain_community

In [ ]:
import os
import json
import time
from openai import OpenAI  #LLM

client = OpenAI(api_key=API_KEY)


In [ ]:
# ============================================================
# 2.1) LLM Automatic Labeling - few-shot prompting without Regex_based rules
# ============================================================
def get_llm_labels(target_text):
    """
    target_text: The sentence from the 1700 unlabeled records
    annotated_complete: The 500 human-annotated reference records
    """

    # 1. Prepare examples: Pick representative samples from the 300
    # example_samples = df_annotated.sample(5)

    # 2. Construct the Prompt
    instruction = """
    	Role: Professional Educational Data Annotator

    	Task: Procedural labeling of student prompts in the context of Human–AI Co-Programming. Analyze the 'Target Prompt' and label it according to 8 epistemic indicators (1 relevance, 1 epistemic aims and 5 epistemic processes). You must output ONLY a valid JSON object.

    	### Labeling Decision Logic:
    	1. Relevance Check: Is the prompt related to learning CS knowledge or programming? If NO, set 'aim_relevance' to 0 and all other labels to 0. If YES, set 'aim_relevance' to 1. If ‘aim_relevance = 0’ all other labels should be 0.

    	2. Aim Classification (Mutual Exclusive Rule):
    	- *understanding_aims*: Set to 1 if the student's inquiry demonstrates a "Strategic or Metacognitive" intent rather than just a "Task Completion" intent. Otherwise set to 0 if the student asks for direct code generation, or debugging help focused on completing the immediate task.

    	3. Specific Process Identification (Check all that apply):
    	- *proc_outsourcing*: Providing assignment questions, or codes, or error messages without any interpretation or specific questions. The student primarily delegates cognitive work to the AI without articulating their own reasoning, interpretation, or specific inquiry. Prompts in this category present tasks, code, or error messages with minimal reflection, indicating cognitive offloading rather than active problem formulation. The core criterion is whether the student demonstrates independent thinking or problem analysis prior to engaging the AI.
    	- *proc_explanation*: The student seeks conceptual or procedural understanding, such as explanations of programming concepts, methods, or how to complete an assignment. Prompts focus on acquiring knowledge about what something is or how to do something, without necessarily proposing or evaluating a solution. The core criterion is the intent to build understanding rather than to validate an existing idea.
    	- *proc_verification*: Reporting errors, asking for code checks, or debugging. The student presents an idea, solution attempt, or hypothesis and asks the AI to check, confirm, or diagnose its correctness. The core criterion is whether the student is seeking confirmation, validation, or corrective feedback on their own reasoning or code.
    	- *proc_prompt*: The student explicitly regulates or refines the AI’s behavior by commenting on prior outputs and requesting changes in style, clarity, scope, or presentation. Prompts in this category demonstrate strategic interaction with the AI to improve response quality through added constraints, context, or delivery preferences. Label a prompt as proc_prompt if the student’s primary intent is to control or refine the AI’s behavior or output characteristics, rather than to obtain explanations, verify correctness, or delegate problem solving.
    	- *proc_justification*: The student engages in higher-order inquiry by questioning assumptions, comparing alternatives, or asking for reasons, trade-offs, or conditional strategies. These prompts reflect epistemic curiosity, agency, and reflective thinking, often extending beyond immediate task completion toward deeper understanding or generalization. The core criterion is whether the student seeks why, when, or under what conditions a solution or approach is appropriate.Label proc_justification when the prompt primarily seeks rationales, comparisons, or conditional strategy (why/when/which approach and under what constraints), indicating deeper inquiry beyond immediate task completion.

    	### JSON Schema Requirement:
    	{
      	"aim_relevance": int,	// 0 or 1
      	"aim_understanding": int,   // 0 or 1
      	"proc_outsourcing": int, // 0 or 1
      	"proc_explanation": int, // 0 or 1
      	"proc_verification": int,// 0 or 1
      	"proc_prompt": int,   	// 0 or 1
      	"proc_justification": int   // 0 or 1
      	}

    	--- EXAMPLES ---
    	--- Example 1---
    	Prompt: "this is my current main.py:
    	from autocomplete import Autocomplete
    	from utilities import read_file, create_gui

    	autocomplete_engine = Autocomplete()
    	filename = 'genZ.txt'
    	read_file(filename, autocomplete_engine)
    	create_gui(autocomplete_engine)
    	suggest_bfs('li')"

    	Labels: {"aim_relevance": 1, "aim_understanding": 0, "proc_outsourcing": 1, "proc_explanation": 0, "proc_verification": 1, "proc_prompt": 0, "proc_justification": 0}

    	--- Example 2---
    	Prompt: "Explain here what differences did you see in the suggestions generated when you used BFS vs DFS vs UCS. BFS : this is the question im supposed to answer "

    	Labels: {"aim_relevance": 1, "aim_understanding": 0, "proc_outsourcing": 1, "proc_explanation": 0, "proc_verification": 0, "proc_prompt": 0, "proc_justification": 0}

    	--- Example 3---
    	Prompt: "where do I type the suggestions? I cant find a place, and how do I realise which search algorithm to use??"
    	Labels: {"aim_relevance": 1, "aim_understanding": 1, "proc_outsourcing": 0, "proc_explanation": 1, "proc_verification": 0, "proc_prompt": 0, "proc_justification": 1}

    	--- Example 4---
      Prompt: "Thanks for the help. Could you clarify what this line of code does:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)?
I understand that we are splitting the data into a testing and training set. Then we use the training set to create our model. But where is the testing set being used? Is it for cross validation? Or for polynomial regression?"
    	Labels: {"aim_relevance": 1, "aim_understanding": 1, "proc_outsourcing": 0, "proc_explanation": 1, "proc_verification": 0, "proc_prompt": 0, "proc_justification": 1}

    	--- Example 5---
    	Prompt: "The task type is text generation. I'm looking to start with a simple model. The computational resources are somewhat small. The first target sequence length is around 2500 characters long, but the second is much longer, around 4200000 characters. "
    	Labels: {"aim_relevance": 1, "aim_understanding": 1, "proc_outsourcing": 0, "proc_explanation": 0, "proc_verification": 0, "proc_prompt": 1, "proc_justification": 1}

    	--- Example 6---
    	Prompt: "If I'm trying to implement a function that generates n tables from a given document that count the frequencies of n-grams (1-grams for the first table, 2-grams for the next, etc.), would this code work for that purpose?

	tables: list[dict[str, int]] = [{} for i in range(n)]
	doclen = len(document)

	for i in range(doclen):
    	for j in range(n):
        	if i + j < doclen:
            	ngram: str = document[i:i + j + 1]

            	if ngram in tables[j]:
                	tables[j][ngram] += 1
            	else:
                	tables[j][ngram] = 1

	return tables"
    	Labels: {"aim_relevance": 1, "aim_understanding": 0, "proc_outsourcing": 0, "proc_explanation": 0, "proc_verification": 1, "proc_prompt": 0, "proc_justification": 0}

    	--- Example 7---
  	  Prompt: “oops that was the wrong code snippet. This is the correct one I used
# Imports section
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split”
    	Labels: {"aim_relevance": 1, "aim_understanding": 0, "proc_outsourcing": 0, "proc_explanation": 0, "proc_verification": 1, "proc_prompt": 1, "proc_justification": 0}

  --- Example 8---
  Prompt: “have you wished pawan a happy birthday today”
  Labels: {"aim_relevance": 0, "aim_understanding": 0, "proc_outsourcing": 0, "proc_explanation": 0, "proc_verification": 1, "proc_prompt": 0, "proc_justification": 0}

"""
    # 3. Final string sent to LLM
    full_prompt = instruction + f"Target Prompt: {target_text}\nLabels:"

    # 4. API Call
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": full_prompt}],
        response_format={ "type": "json_object" }
    )

    return response.choices[0].message.content

In [ ]:
# ============================================================
# 2.2) LLM Automatic Labeling - few-shot prompting with Regex_based rules
# ============================================================
def get_llm_labels_regex(target_text):
    """
    target_text: The sentence from the 1700 unlabeled records
    annotated_complete: The 500 human-annotated reference records
    """

    # 1. Prepare examples: Pick representative samples from the 300
    # example_samples = df_annotated.sample(5)

    # 2. Construct the Prompt
    instruction = """Role: Professional Educational Data Annotator
    	Task: Procedural labeling of student prompts in the context of Human–AI Co-Programming. Analyze the 'Target Prompt' and label it according to 8 epistemic indicators (1 relevance, 1 epistemic aims and 5 epistemic processes). You must output ONLY a valid JSON object.

    	### Labeling Decision Logic:
    	1. Relevance Check: Is the prompt related to learning CS knowledge or programming? If NO, set 'aim_relevance' to 0 and all other labels to 0. If YES, set 'aim_relevance' to 1. If ‘aim_relevance = 0’ all other labels should be 0.

    	2. Aim Classification (Mutual Exclusive Rule):
    	- *understanding_aims*: Set to 1 if the student's inquiry demonstrates a "Strategic or Metacognitive" intent rather than just a "Task Completion" intent. Otherwise set to 0 if the student asks for direct code generation, or debugging help focused on completing the immediate task.
High Probability Indicators (Key Patterns):
Comparative & Critical Thinking: Using contrast words (e.g., 'but', 'however', 'differ', 'difference', 'better', 'easier', 'best', 'i think', 'instead') to evaluate solutions.
Observation & Evidence: Reporting specific findings or anomalies (e.g., 'I observe', 'I found', 'I can see', 'you were unable to', 'hallucination').
Hypothetical & Exploratory Reasoning for justification: Asking about possibilities or limits (e.g., ‘why’, 'what if', 'what would', 'could it be', 'wouldn't', 'is it reasonable', 'does it make sense', 'analy*', 'intuition').
Efficiency & Strategy: Focusing on the efficiency or usefulness of knowledge during the process (e.g., 'efficient', 'too much', 'by hand', 'easier', 'difficult', 'curious').
Learning Self-Regulation: Proposing their own next steps or seeking justification (e.g., "I'm going to", 'suggest', 'ask', 'rational').


     	3. Specific Process Identification (Check all that apply):
    	- *proc_outsourcing*: Providing assignment questions, or codes, or error messages without any interpretation or specific questions. The student primarily delegates cognitive work to the AI without articulating their own reasoning, interpretation, or specific inquiry. Prompts in this category present tasks, code, or error messages with minimal reflection, indicating cognitive offloading rather than active problem formulation. The core criterion is whether the student demonstrates independent thinking or problem analysis prior to engaging the AI.
High Probability Indicators (Key Patterns):
Direct Task Delegation Without Problem Framing: (e.g., “complete this,” “solve the following,” “do this problem”, “help with”, “fix”, “make”); Referential task handoff (e.g., “this is the question,” “here is the problem,” “this is my function”). Absence of metacognitive framing (e.g., no mention of what the student tried, thought, or found confusing)
Uncontextualized Copy–Paste Artifacts: Raw code blocks or snippets without explanation (e.g., imports, function definitions, comments, ‘#’); Assignment-style formatting or capitalization patterns inconsistent with natural student questioning; Lack of personalization (e.g., no reference to what the student understands, intends, or expects)


    	- *proc_explanation*: The student seeks conceptual or procedural understanding, such as explanations of programming concepts, methods, or how to complete an assignment. Prompts focus on acquiring knowledge about what something is or how to do something, without necessarily proposing or evaluating a solution. The core criterion is the intent to build understanding rather than to validate an existing idea.
High Probability Indicators (Key Patterns):
Conceptual Explanation Requests: Definition-seeking phrases (e.g., what is, what does … mean, meaning of, could you explain); Requests for intuition or understanding (e.g., why does, how does it work, what’s the idea behind)
Procedural Explanation Requests: Procedure-seeking phrases (e.g., how to, how can I, how do you, how is … achieved); Step-oriented framing (e.g., what are the steps, how should I approach, how does this part work)


    	- *proc_verification*: Reporting errors, asking for code checks, or debugging. The student presents an idea, solution attempt, or hypothesis and asks the AI to check, confirm, or diagnose its correctness. The core criterion is whether the student is seeking confirmation, validation, or corrective feedback on their own reasoning or code.
High Probability Indicators (Key Patterns):
Code Verification Requests: Verification-oriented verbs or questions (e.g., check, verify, correct, confirm, diagnose, does this work, is this valid/right, what’s wrong with my code, would this work, should/can this…); Referential framing that anchors the question to the student’s own attempt (e.g., here is my code, this is what I wrote, I tried…); Error traces or messages (e.g., SyntaxError, stack traces, error logs)
Knowledge or Concept Verification:Epistemic stance markers (e.g., I think…, my understanding is…); Confirmation-seeking questions (e.g., is this correct, should it be, can X do Y, A or B, does it make sense)


    	- *proc_prompt*: The student explicitly regulates or refines the AI’s behavior by commenting on prior outputs and requesting changes in style, clarity, scope, or presentation. Prompts in this category demonstrate strategic interaction with the AI to improve response quality through added constraints, context, or delivery preferences. Label a prompt as proc_prompt if the student’s primary intent is to control or refine the AI’s behavior or output characteristics, rather than to obtain explanations, verify correctness, or delegate problem solving.
High Probability Indicators (Key Patterns):
Output Regulation and Refinement:Evaluation of prior output (e.g., too much, not clear, you aren’t, your response/answer/result);Style and formatting constraints (e.g., concise, simple, clear, rearrange, stronger, write in 1–2 sentences, 3–4 lines, continuous paragraph)
Interaction and Process Control: Temporal or pacing directives (e.g., wait, slow down, take a second to think); Task framing or sequencing (e.g., your task, next step, start from, don’t solve anything for me)
Contextual Refinement Based on Student Work: Self-referential grounding (e.g., I am working on, my approach/essay/code/function, here is, I have); Clarification or correction of intent (e.g., I mean, note/consider, change/refine)
    	- *proc_justification*: The student engages in higher-order inquiry by questioning assumptions, comparing alternatives, or asking for reasons, trade-offs, or conditional strategies. These prompts reflect epistemic curiosity, agency, and reflective thinking, often extending beyond immediate task completion toward deeper understanding or generalization. The core criterion is whether the student seeks why, when, or under what conditions a solution or approach is appropriate. Label proc_justification when the prompt primarily seeks rationales, comparisons, or conditional strategy (why/when/which approach and under what constraints), indicating deeper inquiry beyond immediate task completion.
High Probability Indicators (Key Patterns):
Reasoning and Rationale Seeking: e,g,(why, intuition, make sense, justify, prove, show evidence, clarify, identify, essential/important)and validity language (e.g., reasonable, sound, robust, rational, valid)
Conditional and Counterfactual Reasoning: Conditionals(e.g., what if, how about, under what conditions, would it/should it/could it be); Failure-triggered inquiry (e.g., that does not work, what happens if, why not)
Alternatives, Comparisons, and Trade-offs: Comparative framing (e.g., better/best, easier, more efficient, difference between, A vs B, A or B); Contrast markers (e.g., but, however, though); Substitution (e.g., can I do … instead, without …, is there another way)
Agency and Reflective Stance: Self-positioning (e.g., I think…, I will do… by hand, my approach is…); Critical stance for disagreement or challenge (e.g., but wouldn’t it…, shouldn’t it…, incorrect/wrong)



    	### JSON Schema Requirement:
    	{
      	"aim_relevance": int,	// 0 or 1
      	"aim_understanding": int,   // 0 or 1
      	"proc_outsourcing": int, // 0 or 1
      	"proc_explanation": int, // 0 or 1
      	"proc_verification": int,// 0 or 1
      	"proc_prompt": int,   	// 0 or 1
      	"proc_justification": int   // 0 or 1
      	}

    	--- EXAMPLES ---
    	--- Example 1---
    	Prompt: "this is my current main.py:
    	from autocomplete import Autocomplete
    	from utilities import read_file, create_gui

    	autocomplete_engine = Autocomplete()
    	filename = 'genZ.txt'
    	read_file(filename, autocomplete_engine)
    	create_gui(autocomplete_engine)
    	suggest_bfs('li')"

    	Labels: {"aim_relevance": 1, "aim_understanding": 0, "proc_outsourcing": 1, "proc_explanation": 0, "proc_verification": 1, "proc_prompt": 0, "proc_justification": 0}

    	--- Example 2---
    	Prompt: "Explain here what differences did you see in the suggestions generated when you used BFS vs DFS vs UCS. BFS : this is the question im supposed to answer "

    	Labels: {"aim_relevance": 1, "aim_understanding": 0, "proc_outsourcing": 1, "proc_explanation": 0, "proc_verification": 0, "proc_prompt": 0, "proc_justification": 0}

    	--- Example 3---
    	Prompt: "where do I type the suggestions? I cant find a place, and how do I realise which search algorithm to use??"
    	Labels: {"aim_relevance": 1, "aim_understanding": 1, "proc_outsourcing": 0, "proc_explanation": 1, "proc_verification": 0, "proc_prompt": 0, "proc_justification": 1}

    	--- Example 4---
    	Prompt: "Thanks for the help. Could you clarify what this line of code does:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)?
I understand that we are splitting the data into a testing and training set. Then we use the training set to create our model. But where is the testing set being used? Is it for cross validation? Or for polynomial regression?"
    	Labels: {"aim_relevance": 1, "aim_understanding": 1, "proc_outsourcing": 0, "proc_explanation": 1, "proc_verification": 0, "proc_prompt": 0, "proc_justification": 1}

    	--- Example 5---
    	Prompt: "The task type is text generation. I'm looking to start with a simple model. The computational resources are somewhat small. The first target sequence length is around 2500 characters long, but the second is much longer, around 4200000 characters. "
    	Labels: {"aim_relevance": 1, "aim_understanding": 1, "proc_outsourcing": 0, "proc_explanation": 0, "proc_verification": 0, "proc_prompt": 1, "proc_justification": 1}

    	--- Example 6---
    	Prompt: "If I'm trying to implement a function that generates n tables from a given document that count the frequencies of n-grams (1-grams for the first table, 2-grams for the next, etc.), would this code work for that purpose?

	tables: list[dict[str, int]] = [{} for i in range(n)]
	doclen = len(document)

	for i in range(doclen):
    	for j in range(n):
        	if i + j < doclen:
            	ngram: str = document[i:i + j + 1]

            	if ngram in tables[j]:
                	tables[j][ngram] += 1
            	else:
                	tables[j][ngram] = 1

	return tables"
    	Labels: {"aim_relevance": 1, "aim_understanding": 0, "proc_outsourcing": 0, "proc_explanation": 0, "proc_verification": 1, "proc_prompt": 0, "proc_justification": 0}

    	--- Example 7---
  	  Prompt: “oops that was the wrong code snippet. This is the correct one I used
# Imports section
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split”
    	Labels: {"aim_relevance": 1, "aim_understanding": 0, "proc_outsourcing": 0, "proc_explanation": 0, "proc_verification": 1, "proc_prompt": 1, "proc_justification": 0}

  --- Example 8---
  Prompt: “have you wished pawan a happy birthday today”
  Labels: {"aim_relevance": 0, "aim_understanding": 0, "proc_outsourcing": 0, "proc_explanation": 0, "proc_verification": 1, "proc_prompt": 0, "proc_justification": 0}
"""

    # 3. Final string sent to LLM
    full_prompt = instruction + f"Target Prompt: {target_text}\nLabels:"

    # 4. API Call
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": full_prompt}],
        response_format={ "type": "json_object" }
    )

    return response.choices[0].message.content

In [ ]:
import random

# call few-shot prompting function for no regex_based principles
# load annotated data
df_annotated = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/anno_complete.xlsx")  #/content/drive/MyDrive/Colab Notebooks/anno_complete.xlsx
df_unlabeled = pd.read_excel("df_sample.xlsx")

target_cols = [
    'aim_relevance', 'aim_understanding',
    'proc_outsourcing', 'proc_explanation', 'proc_verification',
    'proc_prompt', 'proc_justification'
]

# --- batch labeling ---

def batch_auto_label(df_unlabeled, df_annotated, batch_size=50, per_call_sleep=0.2):

    df_unlabeled = df_unlabeled.copy()
    df_unlabeled.columns = df_unlabeled.columns.str.strip()

    # check if conversation_id can be used
    if "conversation_id" not in df_unlabeled.columns:
        if df_unlabeled.index.name == "conversation_id":
            df_unlabeled = df_unlabeled.reset_index()
        else:
            raise KeyError(f"conversation_id not found. columns={df_unlabeled.columns.tolist()}")

    # set results and errors
    results, errors = [], []

    # range for batch
    n = len(df_unlabeled)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = df_unlabeled.iloc[start:end]
        print(f"Processing batch {start//batch_size + 1} ({start}-{end-1})...")

        for _, row in batch.iterrows():
            cid = row.get("conversation_id")
            target_text = row["prompt"]
            try:
                response_text = get_llm_labels(target_text) ####
                #print(response_text)
                #json_str = _extract_json(response_text)
                label_data = json.loads(response_text)

                # the key for matching is conversation_id
                label_data["conversation_id"] = cid
                results.append(label_data)
                #print(f"Current results:{results}")

            except Exception as e:
                errors.append({
                    "conversation_id": cid,
                    "error": str(e),
                })
                print(errors)
            # take a small sleep after each batch
            time.sleep(per_call_sleep)


    df_auto_labeled = pd.DataFrame(results)
    df_errors = pd.DataFrame(errors)

    # --- hard guards (very important) ---
    if df_auto_labeled.empty:
        # 全部失败/没append任何label时，给一个空表但带 key 列
        df_auto_labeled = pd.DataFrame(columns=["conversation_id"])
    else:
        df_auto_labeled.columns = df_auto_labeled.columns.str.strip()
        if "conversation_id" not in df_auto_labeled.columns:
            raise KeyError(f"df_auto_labeled missing conversation_id. columns={df_auto_labeled.columns.tolist()}")

    df_final_all = df_unlabeled.merge(
        df_auto_labeled,
        on="conversation_id",
        how="left",
        validate="one_to_one",
    )
    return df_final_all, df_auto_labeled, df_errors

# --- run + save ---
df_final_all, df_auto_labeled, df_errors = batch_auto_label(
    df_unlabeled=df_unlabeled,      # 注意统一变量名
    df_annotated=df_annotated,
    batch_size=50,
    per_call_sleep=0.2,
)

print(df_auto_labeled.head())
#df_final_all.to_csv("labeled_2000_results.csv", index=False)
#df_errors.to_csv("labeled_2000_errors.csv", index=False)
print(f"Done. success={len(df_auto_labeled)}, errors={len(df_errors)}")

Processing batch 1 (0-49)...
Processing batch 2 (50-99)...
Processing batch 3 (100-149)...
Processing batch 4 (150-199)...
Processing batch 5 (200-249)...
Processing batch 6 (250-299)...
Processing batch 7 (300-349)...
Processing batch 8 (350-399)...
Processing batch 9 (400-449)...
Processing batch 10 (450-499)...
Processing batch 11 (500-549)...
Processing batch 12 (550-599)...
Processing batch 13 (600-649)...
Processing batch 14 (650-699)...
Processing batch 15 (700-749)...
Processing batch 16 (750-799)...
Processing batch 17 (800-849)...
Processing batch 18 (850-899)...
Processing batch 19 (900-949)...
Processing batch 20 (950-999)...
Processing batch 21 (1000-1049)...
Processing batch 22 (1050-1099)...
Processing batch 23 (1100-1149)...
Processing batch 24 (1150-1199)...
Processing batch 25 (1200-1249)...
Processing batch 26 (1250-1299)...
Processing batch 27 (1300-1349)...
Processing batch 28 (1350-1399)...
Processing batch 29 (1400-1449)...
Processing batch 30 (1450-1499)...
Proc

In [ ]:
df_unlabeled.head()

,conversation_id,chatId,interactionCount,chatTotalInteractionCount,prompt,response,prompt_word_len,prompt_char_len,n_qmarks,n_#marks,llm_label
0,00f9ebab-7e4c-471a-8623-523812e2bd4d_0,00f9ebab-7e4c-471a-8623-523812e2bd4d,0,20,from collections import deque\nimport heapq\ni...,Your implementation for the `Autocomplete` cla...,358,3871,0,15,"{'label': 'provide_context>Code', 'justificati..."
1,00f9ebab-7e4c-471a-8623-523812e2bd4d_1,00f9ebab-7e4c-471a-8623-523812e2bd4d,1,20,justwhy@vl965-172-31-13-247 assignment-2-searc...,The error message you're encountering indicate...,39,676,0,0,"{'label': 'provide_context>Error Message', 'ju..."
2,00f9ebab-7e4c-471a-8623-523812e2bd4d_2,00f9ebab-7e4c-471a-8623-523812e2bd4d,2,20,how do I run it such that I can see a demo of ...,To run a demonstration of the three search fun...,17,69,1,0,{'label': 'conceptual_questions>Programming To...
3,00f9ebab-7e4c-471a-8623-523812e2bd4d_3,00f9ebab-7e4c-471a-8623-523812e2bd4d,3,20,this is my current main.py:\nfrom autocomplete...,Looking at your current `main.py` and `utiliti...,112,1314,0,0,{'label': 'contextual_questions>Assignment Cla...
4,00f9ebab-7e4c-471a-8623-523812e2bd4d_4,00f9ebab-7e4c-471a-8623-523812e2bd4d,4,20,im not allowed to modify utilities.py,If you are not allowed to modify `utilities.py...,6,37,0,0,{'label': 'contextual_questions>Assignment Cla...


In [ ]:


# ====== inputs ======
# df_annotated = pd.read_excel("anno_complete.xlsx")
# df_final_all = pd.read_csv("labeled_2000_results.csv")

# load annotated data
df_annotated = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/anno_complete.xlsx")  #/content/drive/MyDrive/Colab Notebooks/anno_complete.xlsx
df_final_all = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/labeled_2000_results.csv")


KEY = "conversation_id"

gold_cols = [
    "aim_relevance_0",
    "understanding_aim_0",
    "proc_outsourcing_0",
    "proc_explanation_0",
    "proc_verification_0",
    "proc_prompt_0",
    "proc_justification_0",
]

pred_cols = [
    "aim_relevance",
    "aim_understanding",
    "proc_outsourcing",
    "proc_explanation",
    "proc_verification",
    "proc_prompt",
    "proc_justification",
]

col_map = dict(zip(gold_cols, pred_cols))

# ====== sanity checks ======
missing_gold = [c for c in [KEY] + gold_cols if c not in df_annotated.columns]
missing_pred = [c for c in [KEY] + pred_cols if c not in df_final_all.columns]
if missing_gold:
    raise KeyError(f"Missing in df_annotated: {missing_gold}")
if missing_pred:
    raise KeyError(f"Missing in df_final_all: {missing_pred}")

# focus only on target labels
gold = df_annotated[[KEY] + gold_cols].drop_duplicates(subset=[KEY]).copy()
pred = df_final_all[[KEY] + pred_cols].drop_duplicates(subset=[KEY]).copy()

# ====== merge (inner: conversation_id) ======
df = gold.merge(pred, on=KEY, how="inner", validate="one_to_one")
print("Rows for evaluation:", len(df))

# ====== normalize to 0/1 int (handles float/str/bool) ======
def to01(s):
    s = pd.to_numeric(s, errors="coerce")
    # if the value is not 0 or 1, keep it as NaN
    return s

for g, p in col_map.items():
    df[g] = to01(df[g])
    df[p] = to01(df[p])

# ====== per-column accuracy + confusion ======
rows = []
for g, p in col_map.items():
    valid = df[[g, p]].dropna()
    if len(valid) == 0:
        rows.append({
            "gold_col": g, "pred_col": p,
            "n": 0, "accuracy": np.nan,
            "tp": 0, "tn": 0, "fp": 0, "fn": 0
        })
        continue

    y = valid[g].astype(int)
    yhat = valid[p].astype(int)

    acc = (y == yhat).mean()
    tp = int(((y == 1) & (yhat == 1)).sum())
    tn = int(((y == 0) & (yhat == 0)).sum())
    fp = int(((y == 0) & (yhat == 1)).sum())
    fn = int(((y == 1) & (yhat == 0)).sum())

    rows.append({
        "gold_col": g,
        "pred_col": p,
        "n": len(valid),
        "accuracy": acc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    })

report = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
print("\nPer-column report:")
print(report[["gold_col", "pred_col", "n", "accuracy", "tp", "tn", "fp", "fn"]])

# ====== overall accuracy (micro + macro) ======
matches = []
counts = []
for g, p in col_map.items():
    valid = df[[g, p]].dropna()
    if len(valid) == 0:
        continue
    matches.append((valid[g].astype(int) == valid[p].astype(int)).sum())
    counts.append(len(valid))

micro_acc = (sum(matches) / sum(counts)) if counts else np.nan
macro_acc = report["accuracy"].mean()

print(f"\nOverall micro accuracy: {micro_acc:.4f}")
print(f"Overall macro accuracy: {macro_acc:.4f}")

# ====== optional: save report ======
report.to_csv("llm_vs_gold_accuracy_report.csv", index=False)

Rows for evaluation: 499

Per-column report:
               gold_col            pred_col    n  accuracy   tp   tn   fp  fn
5         proc_prompt_0         proc_prompt  499  0.887776    4  439    7  49
0       aim_relevance_0       aim_relevance  499  0.879760  430    9    0  60
6  proc_justification_0  proc_justification  499  0.877756   23  415   41  20
4   proc_verification_0   proc_verification  499  0.833667   39  377   61  22
3    proc_explanation_0    proc_explanation  499  0.787575  146  247   70  36
2    proc_outsourcing_0    proc_outsourcing  499  0.747495  109  264   52  74
1   understanding_aim_0   aim_understanding  499  0.709419   42  312  110  35

Overall micro accuracy: 0.8176
Overall macro accuracy: 0.8176


In [ ]:
import random
import time

# call few-shot prompting function for including regex_based principles
# load annotated data
#df_annotated = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/anno_complete.xlsx")  #/content/drive/MyDrive/Colab Notebooks/anno_complete.xlsx
df_unlabeled = pd.read_excel("df_sample.xlsx")

target_cols = [
    'aim_relevance', 'aim_understanding',
    'proc_outsourcing', 'proc_explanation', 'proc_verification',
    'proc_prompt', 'proc_justification'
]

# --- batch labeling ---

def batch_auto_label(df_unlabeled, batch_size=50, per_call_sleep=0.2):

    df_unlabeled = df_unlabeled.copy()
    df_unlabeled.columns = df_unlabeled.columns.str.strip()

    # check if conversation_id can be used
    if "conversation_id" not in df_unlabeled.columns:
        if df_unlabeled.index.name == "conversation_id":
            df_unlabeled = df_unlabeled.reset_index()
        else:
            raise KeyError(f"conversation_id not found. columns={df_unlabeled.columns.tolist()}")

    # set results and errors
    results, errors = [], []

    # range for batch
    n = len(df_unlabeled)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = df_unlabeled.iloc[start:end]
        print(f"Processing batch {start//batch_size + 1} ({start}-{end-1})...")

        for _, row in batch.iterrows():
            cid = row.get("conversation_id")
            target_text = row["prompt"]
            try:
                response_text = get_llm_labels_regex(target_text)
                #print(response_text)
                #json_str = _extract_json(response_text)
                label_data = json.loads(response_text)

                # the key for matching is conversation_id
                label_data["conversation_id"] = cid
                results.append(label_data)

                #print(f"Current results:{results}")

            except Exception as e:
                errors.append({
                    "conversation_id": cid,
                    "error": str(e),
                })
                print(errors)
            # take a small sleep after each batch
            time.sleep(per_call_sleep)


    df_auto_labeled_regex= pd.DataFrame(results)
    df_errors_regex = pd.DataFrame(errors)

    # hard guards
    if df_auto_labeled_regex.empty:
        # if all fail, then return an empty column with a key
        df_auto_labeled_regex = pd.DataFrame(columns=["conversation_id"])
    else:
        df_auto_labeled_regex.columns = df_auto_labeled_regex.columns.str.strip()
        if "conversation_id" not in df_auto_labeled_regex.columns:
            raise KeyError(f"df_auto_labeled missing conversation_id. columns={df_auto_labeled_regex.columns.tolist()}")

    df_final_all_regex = df_unlabeled.merge(
        df_auto_labeled_regex,
        on="conversation_id",
        how="left",
        validate="one_to_one",
    )
    return df_final_all_regex, df_auto_labeled_regex, df_errors_regex

# run and save
df_final_all_regex, df_auto_labeled_regex, df_errors_regex = batch_auto_label(
    df_unlabeled=df_unlabeled,
    batch_size=50,
    per_call_sleep=0.2,
)

print(df_auto_labeled_regex.head())

df_final_all_regex.to_csv("labeled_regex_2000_results.csv", index=False)
print(f"Done. success={len(df_auto_labeled_regex)}, errors={len(df_errors_regex)}")

from google.colab import files
files.download("labeled_regex_2000_results.csv")




Processing batch 1 (0-49)...
Processing batch 2 (50-99)...
Processing batch 3 (100-149)...
Processing batch 4 (150-199)...
Processing batch 5 (200-249)...
Processing batch 6 (250-299)...
Processing batch 7 (300-349)...
Processing batch 8 (350-399)...
Processing batch 9 (400-449)...
Processing batch 10 (450-499)...
Processing batch 11 (500-549)...
Processing batch 12 (550-599)...
Processing batch 13 (600-649)...
Processing batch 14 (650-699)...
Processing batch 15 (700-749)...
Processing batch 16 (750-799)...
Processing batch 17 (800-849)...
Processing batch 18 (850-899)...
Processing batch 19 (900-949)...
Processing batch 20 (950-999)...
Processing batch 21 (1000-1049)...
Processing batch 22 (1050-1099)...
Processing batch 23 (1100-1149)...
Processing batch 24 (1150-1199)...
Processing batch 25 (1200-1249)...
Processing batch 26 (1250-1299)...
Processing batch 27 (1300-1349)...
Processing batch 28 (1350-1399)...
Processing batch 29 (1400-1449)...
Processing batch 30 (1450-1499)...
Proc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:


# ====== inputs ======
# 已经在内存里就不用读；否则取消注释读取：
# df_annotated = pd.read_excel("anno_complete.xlsx")
# df_final_all = pd.read_csv("labeled_2000_results.csv")

# load annotated data
df_annotated = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/anno_complete.xlsx")  #/content/drive/MyDrive/Colab Notebooks/anno_complete.xlsx
df_final_all = pd.read_csv("labeled_regex_2000_results.csv")


KEY = "conversation_id"

gold_cols = [
    "aim_relevance_0",
    "understanding_aim_0",
    "proc_outsourcing_0",
    "proc_explanation_0",
    "proc_verification_0",
    "proc_prompt_0",
    "proc_justification_0",
]

pred_cols = [
    "aim_relevance",
    "aim_understanding",
    "proc_outsourcing",
    "proc_explanation",
    "proc_verification",
    "proc_prompt",
    "proc_justification",
]

col_map = dict(zip(gold_cols, pred_cols))

# ====== sanity checks ======
missing_gold = [c for c in [KEY] + gold_cols if c not in df_annotated.columns]
missing_pred = [c for c in [KEY] + pred_cols if c not in df_final_all.columns]
if missing_gold:
    raise KeyError(f"Missing in df_annotated: {missing_gold}")
if missing_pred:
    raise KeyError(f"Missing in df_final_all: {missing_pred}")

# focus only on target labels
gold = df_annotated[[KEY] + gold_cols].drop_duplicates(subset=[KEY]).copy()
pred = df_final_all[[KEY] + pred_cols].drop_duplicates(subset=[KEY]).copy()

# ====== merge (inner: conversation_id) ======
df = gold.merge(pred, on=KEY, how="inner", validate="one_to_one")
print("Rows for evaluation:", len(df))

# ====== normalize to 0/1 int (handles float/str/bool) ======
def to01(s):
    s = pd.to_numeric(s, errors="coerce")
    # if the value is not 0 or 1, keep it as NaN
    return s

for g, p in col_map.items():
    df[g] = to01(df[g])
    df[p] = to01(df[p])

# ====== per-column accuracy + confusion ======
rows = []
for g, p in col_map.items():
    valid = df[[g, p]].dropna()
    if len(valid) == 0:
        rows.append({
            "gold_col": g, "pred_col": p,
            "n": 0, "accuracy": np.nan,
            "tp": 0, "tn": 0, "fp": 0, "fn": 0
        })
        continue

    y = valid[g].astype(int)
    yhat = valid[p].astype(int)

    acc = (y == yhat).mean()
    tp = int(((y == 1) & (yhat == 1)).sum())
    tn = int(((y == 0) & (yhat == 0)).sum())
    fp = int(((y == 0) & (yhat == 1)).sum())
    fn = int(((y == 1) & (yhat == 0)).sum())

    rows.append({
        "gold_col": g,
        "pred_col": p,
        "n": len(valid),
        "accuracy": acc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    })

report = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
print("\nPer-column report:")
print(report[["gold_col", "pred_col", "n", "accuracy", "tp", "tn", "fp", "fn"]])

# ====== overall accuracy (micro + macro) ======
matches = []
counts = []
for g, p in col_map.items():
    valid = df[[g, p]].dropna()
    if len(valid) == 0:
        continue
    matches.append((valid[g].astype(int) == valid[p].astype(int)).sum())
    counts.append(len(valid))

micro_acc = (sum(matches) / sum(counts)) if counts else np.nan
macro_acc = report["accuracy"].mean()

print(f"\nOverall micro accuracy: {micro_acc:.4f}")
print(f"Overall macro accuracy: {macro_acc:.4f}")

# ====== optional: save report ======
report.to_csv("llm_vs_gold_accuracy_report.csv", index=False)

Rows for evaluation: 499

Per-column report:
               gold_col            pred_col    n  accuracy   tp   tn  fp  fn
5         proc_prompt_0         proc_prompt  498  0.897590    9  438   7  44
6  proc_justification_0  proc_justification  498  0.895582   27  419  36  16
0       aim_relevance_0       aim_relevance  498  0.871486  425    9   0  64
4   proc_verification_0   proc_verification  498  0.845382   44  377  60  17
3    proc_explanation_0    proc_explanation  498  0.843373  142  278  39  39
2    proc_outsourcing_0    proc_outsourcing  498  0.809237  128  275  40  55
1   understanding_aim_0   aim_understanding  498  0.809237   37  366  55  40

Overall micro accuracy: 0.8531
Overall macro accuracy: 0.8531


In [ ]:
# Batch Processing Loop
batch_size = 100
for i in range(0, len(df_sample), batch_size):
  # Get the current batch
  batch = df_sample.iloc[i: i + batch_size]
  print(f"Processing batch {i//batch_size + 1}...")

  for index, row in batch.iterrows():
    # traget_text is the content in the prompt column
    target_text = row['prompt']
    try:
      # Call the previously defined labeling function
      response_json = get_llm_labels(target_text) # get_llm_labels_noregex

      # Ensure ID is included for merging
      df_annotated["conversation_id"] = row["conversation_id"]
      # json_file need to be restructure to match, copy and paste to values in target columms

    except Exception as e:
      print(f"Error at ID {row['conversation_id']}: {e}")
      continue
  # sleep briefly to avoid rate limits
  time.sleep(1)


# Merge Results



# Semi-supervised Learning Approach


In [ ]:
# ============================================================
# 4) Feature Engineering Process
# ============================================================


# rules for procedural aims
procedural_keys = [
        'i can see', 'efficient', 'how is', 'how about', 'difference', 'differ',
        'but', 'though', 'because', 'however', 'what would', "wouldn't", 'why',
        'too much', 'by hand', "i'm going to", 'i observe', 'you were unable to',
        'hallucinat', 'could it be', 'reasonable', 'better', 'easier', 'best', 'difficult'
        'ask', 'suggest', 'question', 'curious'
    ]

# rules for off tasks
# no prompt only copy paste questions

EPISTEMIC_VERBS = [
    "explain","justify","prove","verify","check","confirm","compare","evaluate",
    "reason","cite","source","evidence","assumption","uncertain","uncertainty"
]

MODALS = ["should","could","would","might","may","must"]
HEDGES = ["maybe","perhaps","i think","i guess","not sure","unsure","probably"]

def safe_text(x):
    return "" if not isinstance(x, str) else x

def lexical_diversity(text):
    toks = re.findall(r"\b\w+\b", text.lower())
    if not toks:
        return 0.0
    return len(set(toks)) / len(toks)

def count_terms(text, terms):
    t = text.lower()
    return sum(t.count(w) for w in terms)

def extract_prompt_features(df, text_col="prompt"):
    out = pd.DataFrame(index=df.index)
    txt = df[text_col].astype(str).fillna("")

    out["prompt_word_len"] = txt.apply(lambda s: len(re.findall(r"\b\w+\b", s)))
    out["prompt_char_len"] = txt.str.len()
    out["n_qmarks"] = txt.str.count(r"\?")
    out["n_exclam"] = txt.str.count(r"!")
    out["n_code_marks"] = txt.str.count(r"```") + txt.str.count(r"\bimport\b") + txt.str.count(r";")
    out["lex_div"] = txt.apply(lexical_diversity)

    out["n_epistemic_verbs"] = txt.apply(lambda s: count_terms(s, EPISTEMIC_VERBS))
    out["n_modals"] = txt.apply(lambda s: count_terms(s, MODALS))
    out["n_hedges"] = txt.apply(lambda s: count_terms(s, HEDGES))

    out["has_url"] = txt.str.contains(r"http[s]?://", case=False, regex=True).astype(int)
    out["has_numbers"] = txt.str.contains(r"\d", regex=True).astype(int)
    out["has_compare_words"] = txt.str.contains(r"\b(compare|difference|versus|vs)\b", case=False, regex=True).astype(int)
    out["has_sources_words"] = txt.str.contains(r"\b(source|citation|cite|reference|paper)\b", case=False, regex=True).astype(int)

    return out

feat_A = extract_prompt_features(df_anno, text_col=TEXT_COL)

# Add your regex outputs as *additional features*
regex_feat_cols = [
    "aim_conceptual","aim_procedural",
    "proc_outsourcing","proc_explanation","proc_verification",
    "proc_prompt_improve","proc_regulation"
]
feat_A = pd.concat([feat_A, df_anno[regex_feat_cols].astype(int)], axis=1)

feat_A.head()


# Regex_based Script Feature Engineering

---



In [ ]:
### More feasible plan

# load data
# raw data = df_sample
# annotated data
df_annotated = pd.read_csv('anno_complete.xlsx')


# Labels
target_columns = [
    'aim_relevance', 'aim_conceptual', 'aim_procedural',
    'proc_outsourcing', 'proc_explanation', 'proc_verification',
    'proc_prompt_improvement', 'proc_regulation'
]


def advanced_labeling_logic(text):
    """
    Hierarchical Labeling Logic Function
    """
    text = str(text).lower()

    labels = {
        'reg_relevance': 0, 'reg_conceptual': 0, 'reg_procedural': 0,
        'reg_outsourcing': 0, 'reg_explanation': 0, 'reg_verification': 0,
        'reg_prompt_imp': 0, 'reg_regulation': 0
    }

    # Define signals for Procedural Aim
    procedural_keywords = [
        'i can see', 'efficient', 'how is', 'how about', 'difference', 'differ',
        'but', 'though', 'because', 'however', 'what would', "wouldn't", 'why',
        'too much', 'by hand', "i'm going to", 'i observe', 'you were unable to',
        'hallucinat', 'could it be', 'reasonable', 'better', 'easier', 'best',
        'ask', 'suggest', 'question', 'curious'
    ]

    # Procedural Aim / Detecting Procedural Aim
    hit_proc_keywords = any(k in text for k in procedural_keywords)
    word_count = len(text.split())

    if hit_proc_keywords or word_count > 20:
        labels['reg_procedural'] = 1
        labels['reg_relevance'] = 1 # Procedural 必然是相关的 / Procedural is naturally relevant

    # Sub-category detection (Only if Procedural)
    if labels['aim_procedural'] == 1:
        # Verification
        if re.search(r'(verify|check|correct|right|hallucinat)', text):
            labels['proc_verification'] = 1
        # Explanation
        if re.search(r'(why|because|reason|how come)', text):
            labels['proc_explanation'] = 1
        # Prompting
        if re.search(r'(prompt|suggest|ask|how to ask)', text):
            labels['proc_prompt'] = 1
        #  Regulation
        if re.search(r'(rule|limit|regulation|standard)', text):
            labels['proc_regulation'] = 1

    # Core Logic: If not Procedural, then Conceptual
    if labels['aim_procedural'] == 0:
        # Check for conceptual keywords; if none, it's Off-topic
        conceptual_keywords = ['what', 'define', 'concept', 'theory', 'meaning']
        if any(k in text for k in conceptual_keywords):
            labels['aim_conceptual'] = 1
            labels['aim_relevance'] = 1
        else:
            #Off-topic / Neither P nor C, marked as Off-topic
            labels['aim_relevance'] = 0

    return pd.Series(labels)

# Data Processing Workflow 

#  Apply labeling function
# df_unlabeled refers to the 1700 records
auto_labels = df_unlabeled['prompt'].apply(advanced_labeling_logic)
df_result = pd.concat([df_unlabeled[['conversation_id', 'prompt']], auto_labels], axis=1)

# Print descriptive statistics
print("--- 描述性统计报告 (2000条汇总) / Descriptive Statistics Report ---")
print(df_result.iloc[:, 2:].mean()) #Show proportion of each label
### method 1: Keyword matching extraction based on Regex ###
def regex_labeling(text):
    text = str(text).lower()
    labels = {col: 0 for col in target_columns}

    # Rules based on the observation of 300 annotated data
    if re.search(r'(what|define|mean|concept)', text):
        labels['aim_conceptual'] = 1
    if re.search(r'(how to|steps|guide|process)', text):
        labels['aim_procedural'] = 1
    if re.search(r'(check|verify|correct|is this right)', text):
        labels['proc_verification'] = 1
    if re.search(r'(why|explain|reason)', text):
        labels['proc_explanation'] = 1

    return pd.Series(labels)

### Method B: LLM Few-shot Prompt Construction
def generate_llm_prompt(row, examples_df):
    # Learning from examples
    prompt_template = "You are an expert annotator. Based on the user prompt, label 8 categories as 0 or 1.\n"

    # Few-shot examples
    for _, ex in examples_df.sample(3).iterrows():
        prompt_template += f"Prompt: {ex['prompt']}\nLabels: {ex[target_columns].to_dict()}\n---\n"

    prompt_template += f"Now label this one:\nPrompt: {row['prompt']}\nLabels:"
    return prompt_template

# Find unlabeled data
df_unlabeled = df_raw[~df_raw['conversation_id'].isin(df_annotated['conversation_id'])].copy()

# Apply regex model
auto_labels = df_unlabeled['prompt'].apply(regex_labeling)
df_unlabeled[target_columns] = auto_labels

# merge 
df_final = pd.concat([df_annotated, df_unlabeled], axis=0, ignore_index=True)

# Descriptive statistics
summary_stats = df_final[target_columns].mean().sort_values(ascending=False)
print("各维度占比分布:\n", summary_stats)

# Interplay between interactionCount and relevance
correlation = df_final[['interactionCount', 'aim_relevance']].corr()
print("\n相关性矩阵:\n", correlation)

In [ ]:
# ============================================================
# 3) Load + run regex on annotated 300
# ============================================================

import pandas as pd


df_anno = pd.read_excel("anno_complete.xlsx")  # upload to colab first

# REQUIRED columns (change if yours differ)
TEXT_COL = "prompt"         # student prompt text
RESPONSE_COL = "response"   # assistant response text (kept for context)

# Run your regex classifier to add regex-based indicator columns
df_anno = run_regex_classifier(df_anno, text_col=TEXT_COL)

df_anno.head(3)


In [ ]:
# ============================================================
# 4) Feature-rich descriptive modeling (interpretable features)
# ============================================================
import re
import numpy as np

EPISTEMIC_VERBS = [
    "explain","justify","prove","verify","check","confirm","compare","evaluate",
    "reason","cite","source","evidence","assumption","uncertain","uncertainty"
]
MODALS = ["should","could","would","might","may","must"]
HEDGES = ["maybe","perhaps","i think","i guess","not sure","unsure","probably"]

# Epistemic Processes

def safe_text(x):
    return "" if not isinstance(x, str) else x

def lexical_diversity(text):
    toks = re.findall(r"\b\w+\b", text.lower())
    if not toks:
        return 0.0
    return len(set(toks)) / len(toks)

def count_terms(text, terms):
    t = text.lower()
    return sum(t.count(w) for w in terms)

def extract_prompt_features(df, text_col="prompt"):
    out = pd.DataFrame(index=df.index)
    txt = df[text_col].astype(str).fillna("")

    out["prompt_word_len"] = txt.apply(lambda s: len(re.findall(r"\b\w+\b", s)))
    out["prompt_char_len"] = txt.str.len()
    out["n_qmarks"] = txt.str.count(r"\?")
    out["n_exclam"] = txt.str.count(r"!")
    out["n_code_marks"] = txt.str.count(r"```") + txt.str.count(r"\bimport\b") + txt.str.count(r";")
    out["lex_div"] = txt.apply(lexical_diversity)

    out["n_epistemic_verbs"] = txt.apply(lambda s: count_terms(s, EPISTEMIC_VERBS))
    out["n_modals"] = txt.apply(lambda s: count_terms(s, MODALS))
    out["n_hedges"] = txt.apply(lambda s: count_terms(s, HEDGES))

    out["has_url"] = txt.str.contains(r"http[s]?://", case=False, regex=True).astype(int)
    out["has_numbers"] = txt.str.contains(r"\d", regex=True).astype(int)
    out["has_compare_words"] = txt.str.contains(r"\b(compare|difference|versus|vs)\b", case=False, regex=True).astype(int)
    out["has_sources_words"] = txt.str.contains(r"\b(source|citation|cite|reference|paper)\b", case=False, regex=True).astype(int)

    return out

feat_A = extract_prompt_features(df_anno, text_col=TEXT_COL)

# Add your regex outputs as *additional features*
regex_feat_cols = [
    "aim_conceptual","aim_procedural",
    "proc_outsourcing","proc_explanation","proc_verification",
    "proc_prompt_improve","proc_regulation"
]
feat_A = pd.concat([feat_A, df_anno[regex_feat_cols].astype(int)], axis=1)

feat_A.head()


In [ ]:
"""Workflow script (public data -> dedup -> few-shot LLM labeling -> evaluation).

Inputs:
  - turn_df.xlsx            (turn-level public data)
  - anno_complete.xlsx      (human annotations; a subset of turns)

Output:
  - labeled_2000.xlsx       (2000 LLM-labeled rows incl. the 500 human-labeled rows)
  - eval_report.json        (basic agreement metrics vs. human labels)

Notes:
  - If the same conversation_id appears in both sources, we keep annotated rows to avoid duplication/leakage.
  - Few-shot examples should be intentionally curated; this script supports both manual lists and
    a simple stratified sampler (fallback).
"""

from __future__ import annotations

import argparse
import json
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# ---------------------------
# Config
# ---------------------------

ID_COL = "conversation_id"
TEXT_COLS = ["prompt", "response"]

TARGET_COLS = [
    "relevance",
    "aim_relevance",
    "aim_understanding",
    "proc_outsourcing",
    "proc_explanation",
    "proc_verification",
    "proc_prompt",
    "proc_justification",
]

KEEP_COLS = [
    "conversation_id",
    "interactionCount",
    "llm_label",
    "response",
    "prompt",
] + TARGET_COLS


# ---------------------------
# Utilities
# ---------------------------


def read_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    # Normalize column names (strip whitespace)
    df.columns = [str(c).strip() for c in df.columns]
    return df


def ensure_columns(df: pd.DataFrame, needed: List[str], name: str) -> None:
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")


def dedup_keep_annotated(turn_df: pd.DataFrame, anno_df: pd.DataFrame) -> pd.DataFrame:
    """Combine turn_df + anno_df, but for any conversation_id that appears in anno_df,
    keep ONLY the annotated rows for that conversation_id.

    Rationale: avoid double counting / leakage when the same conversation is present in both.
    """
    anno_conv = set(anno_df[ID_COL].dropna().astype(str))
    turn_df = turn_df.copy()
    anno_df = anno_df.copy()

    turn_df[ID_COL] = turn_df[ID_COL].astype(str)
    anno_df[ID_COL] = anno_df[ID_COL].astype(str)

    # Drop rows from turn_df that belong to conversations we have annotations for
    turn_df_filtered = turn_df[~turn_df[ID_COL].isin(anno_conv)]

    # Keep anno_df as authoritative
    combined = pd.concat([turn_df_filtered, anno_df], ignore_index=True)

    # Optional: if anno_df itself has duplicates, drop exact duplicates
    combined = combined.drop_duplicates(subset=[ID_COL] + [c for c in TEXT_COLS if c in combined.columns], keep="first")
    return combined


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def stratified_fewshot_indices(
    anno_df: pd.DataFrame,
    k: int,
    seed: int,
    primary_strata: str = "relevance",
) -> List[int]:
    """Fallback sampler if you don't provide manual few-shot indices.

    This is NOT as good as intentional curation, but helps bootstrap a balanced set.
    It tries to balance by a primary label (default: relevance).
    """
    set_seed(seed)
    if primary_strata not in anno_df.columns:
        return anno_df.sample(n=min(k, len(anno_df)), random_state=seed).index.tolist()

    groups = []
    for _, g in anno_df.groupby(primary_strata, dropna=False):
        groups.append(g)

    per = max(1, k // max(1, len(groups)))
    picks = []
    for g in groups:
        picks.extend(g.sample(n=min(per, len(g)), random_state=seed).index.tolist())

    # Top up if needed
    if len(picks) < min(k, len(anno_df)):
        remaining = anno_df.drop(index=picks, errors="ignore")
        extra = remaining.sample(n=min(k - len(picks), len(remaining)), random_state=seed).index.tolist()
        picks.extend(extra)

    return picks[: min(k, len(anno_df))]


# ---------------------------
# LLM labeling (stub)
# ---------------------------


def build_prompt(fewshot_rows: pd.DataFrame, row: pd.Series) -> str:
    """Build a single prompt for labeling one row.

    Replace this with your final taxonomy + decision rules.
    """
    def fmt_example(r: pd.Series) -> str:
        # Use compact format; avoid leaking conversation_id into model if not needed.
        return (
            "Example\n"
            f"Prompt: {r.get('prompt','')}\n"
            f"Response: {r.get('response','')}\n"
            "Labels (JSON): "
            + json.dumps({c: int(r[c]) if pd.notna(r.get(c)) else None for c in TARGET_COLS}, ensure_ascii=False)
            + "\n"
        )

    shots = "\n".join(fmt_example(r) for _, r in fewshot_rows.iterrows())
    task = (
        "Task\n"
        "Label the following turn with these fields: "
        + ", ".join(TARGET_COLS)
        + ". Output ONLY valid JSON with these keys and integer values (0/1 or as defined).\n\n"
        f"Prompt: {row.get('prompt','')}\n"
        f"Response: {row.get('response','')}\n"
    )

    return shots + "\n" + task


@dataclass
class LLMConfig:
    provider: str
    model: str
    temperature: float = 0.0
    top_p: float = 1.0
    seed: Optional[int] = 1


def llm_label_one(prompt: str, cfg: LLMConfig) -> Dict[str, int]:
    """Stub for calling an LLM and returning parsed JSON.

    You should implement the provider-specific call here (OpenAI/Qwen/Gemini, etc.).
    This function must:
      - send `prompt`
      - enforce temperature=0 for stability
      - return a dict with TARGET_COLS keys

    For now, it returns dummy zeros.
    """
    _ = (prompt, cfg)
    return {c: 0 for c in TARGET_COLS}


def label_dataframe(
    df: pd.DataFrame,
    fewshot_df: pd.DataFrame,
    cfg: LLMConfig,
) -> pd.DataFrame:
    out = df.copy()
    for c in TARGET_COLS:
        if c not in out.columns:
            out[c] = np.nan

    for i, row in out.iterrows():
        # Skip if already labeled (human)
        if all(pd.notna(row.get(c)) for c in TARGET_COLS if c in out.columns):
            continue

        p = build_prompt(fewshot_df, row)
        pred = llm_label_one(p, cfg)
        for c in TARGET_COLS:
            out.at[i, c] = pred.get(c, np.nan)

    return out


# ---------------------------
# Evaluation
# ---------------------------


def agreement_report(gold: pd.DataFrame, pred: pd.DataFrame) -> Dict:
    """Compute simple agreement for rows present in gold by matching on conversation_id + prompt/response."""
    key_cols = [ID_COL] + [c for c in TEXT_COLS if c in gold.columns and c in pred.columns]

    g = gold.copy()
    p = pred.copy()
    g[ID_COL] = g[ID_COL].astype(str)
    p[ID_COL] = p[ID_COL].astype(str)

    merged = g.merge(p, on=key_cols, suffixes=("_gold", "_pred"), how="inner")

    report: Dict[str, Dict[str, float]] = {"n": int(len(merged))}
    for c in TARGET_COLS:
        cg = f"{c}_gold"
        cp = f"{c}_pred"
        if cg not in merged.columns or cp not in merged.columns:
            continue
        m = merged[[cg, cp]].dropna()
        if len(m) == 0:
            report[c] = {"n": 0}
            continue
        acc = float((m[cg].astype(int) == m[cp].astype(int)).mean())
        report[c] = {"n": int(len(m)), "accuracy": acc}

    return report


# ---------------------------
# Main
# ---------------------------


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--turn", default="turn_df.xlsx")
    ap.add_argument("--anno", default="anno_complete.xlsx")
    ap.add_argument("--out", default="labeled_2000.xlsx")
    ap.add_argument("--eval_out", default="eval_report.json")
    ap.add_argument("--n_gold", type=int, default=500)
    ap.add_argument("--n_total", type=int, default=2000)
    ap.add_argument("--fewshot_k", type=int, default=24)
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--provider", default="openai")
    ap.add_argument("--model", default="gpt-4.1-mini")
    args = ap.parse_args()

    set_seed(args.seed)

    turn_df = read_excel(args.turn)
    anno_df = read_excel(args.anno)

    ensure_columns(turn_df, [ID_COL] + TEXT_COLS, "turn_df")
    ensure_columns(anno_df, [ID_COL] + TEXT_COLS, "anno_df")

    # Keep only known cols if present
    for c in KEEP_COLS:
        if c not in turn_df.columns:
            turn_df[c] = np.nan
        if c not in anno_df.columns:
            anno_df[c] = np.nan
    turn_df = turn_df[KEEP_COLS]
    anno_df = anno_df[KEEP_COLS]

    combined = dedup_keep_annotated(turn_df, anno_df)

    # Gold set: take from anno_df (authoritative)
    gold = anno_df.copy()
    if len(gold) > args.n_gold:
        gold = gold.sample(n=args.n_gold, random_state=args.seed)

    # Build pool for 2000 set (must include gold)
    # Exclude gold rows from pool by exact key match
    key_cols = [ID_COL] + TEXT_COLS
    pool = combined.merge(gold[key_cols], on=key_cols, how="left", indicator=True)
    pool = pool[pool["_merge"] == "left_only"].drop(columns=["_merge"])

    need = max(0, args.n_total - len(gold))
    extra = pool.sample(n=min(need, len(pool)), random_state=args.seed) if need else pool.head(0)

    to_label = pd.concat([gold, extra], ignore_index=True)

    # Few-shot examples
    # Option A (recommended): manually specify indices in the gold dataframe.
    manual_indices: List[int] = []  # e.g., [0, 5, 12, ...] in `gold` after sampling

    if manual_indices:
        fewshot = gold.loc[manual_indices]
    else:
        idx = stratified_fewshot_indices(gold, k=args.fewshot_k, seed=args.seed, primary_strata="relevance")
        fewshot = gold.loc[idx]

    cfg = LLMConfig(provider=args.provider, model=args.model, temperature=0.0, top_p=1.0, seed=args.seed)

    labeled = label_dataframe(to_label, fewshot_df=fewshot, cfg=cfg)

    # Evaluate: compare model predictions on gold rows (if you force re-labeling gold)
    # Here we keep gold labels, so agreement will be trivially 1.0.
    # If you want true evaluation, create a copy of gold with targets blanked out before labeling.
    report = agreement_report(gold=gold, pred=labeled)

    labeled.to_excel(args.out, index=False)
    with open(args.eval_out, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"Wrote: {args.out}")
    print(f"Wrote: {args.eval_out}")


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# 3) Regex-based classifier (8 indicators)
# ============================================================

@dataclass
class RegexRules:
    # Aims
    conceptual: List[str]
    procedural: List[str]

    # Processes
    outsourcing: List[str]
    explanation: List[str]
    verification: List[str]
    prompt_improve: List[str]
    regulation: List[str]


DEFAULT_RULES = RegexRules(
    # --- Aims ---
    conceptual=[
        r"\bwhat is\b", r"\bwhat does\b.*\bmean\b", r"\bwhy\b", r"\bhow does\b",
        r"\bconcept\b", r"\bprinciple\b", r"\bintuition\b", r"\bexplain\b.*\bconcept\b"
    ],
    procedural=[
        r"\bhow to\b", r"\bimplement\b", r"\bwrite\b.*\bfunction\b", r"\bdebug\b",
        r"\bfix\b", r"\berror\b", r"\bstack trace\b", r"\bcompile\b", r"\brun\b",
        r"\bpass\b.*\btest\b", r"\boptimize\b"
    ],

    # --- Processes ---
    outsourcing=[
        r"\bjust give me\b", r"\bgive me the answer\b", r"\bfull code\b", r"\bsolve this\b",
        r"\bdo it for me\b", r"\bwrite the entire\b", r"\bcomplete solution\b",
        r"\bhere is the question\b.*\banswer\b"
    ],
    explanation=[
        r"\bexplain\b", r"\bwhy\b", r"\bhow does\b", r"\bwhat is\b", r"\bhelp me understand\b",
        r"\bwalk me through\b"
    ],
    verification=[
        r"\bcheck\b", r"\bverify\b", r"\bconfirm\b", r"\bis this correct\b", r"\bwrong\b", r"\bdoes this work\b",
        r"\brefine\b", r"\bvalidate\b", r"\bedge case\b"
    ],
    prompt_improve=[
        r"\bsimplify\b", r"\brephrase\b", r"\beasier\b", r"\bstep[- ]by[- ]step\b",
        r"\bexplain\b.*\bsimpler\b", r"\bmore clearly\b", r"\bELI5\b"
    ],
    regulation=[
        r"\bnot sure\b", r"\bwhat should i do next\b", r"\bnext step\b", r"\bavoid\b.*\bmistake\b",
        r"\bmore examples\b", r"\bmore evidence\b", r"\bis there a better way\b",
        r"\bhow can i\b.*\bimprove\b", r"\bwhen should i\b", r"\bshould i\b.*\bcheck\b"
    ]
)


def _match_any(patterns: List[str], text: str) -> int:
    if not isinstance(text, str) or not text.strip():
        return 0
    for p in patterns:
        if re.search(p, text, flags=re.IGNORECASE | re.DOTALL):
            return 1
    return 0


def regex_classify_row(text: str, rules: RegexRules = DEFAULT_RULES) -> Dict[str, object]:
    """
    Returns:
      - aim_relevance: on-topic/off-topic (heuristic: if none of the aim patterns match => off-topic)
      - aim_conceptual/procedural: 0/1
      - process labels: 0/1
    """
    conceptual = _match_any(rules.conceptual, text)
    procedural = _match_any(rules.procedural, text)

    # Relevance heuristic: if it triggers either conceptual or procedural => on-topic else off-topic
    relevance = "on-topic" if (conceptual or procedural) else "off-topic"

    out = {
        "aim_relevance": relevance,
        "aim_conceptual": conceptual,
        "aim_procedural": procedural,
        "proc_outsourcing": _match_any(rules.outsourcing, text),
        "proc_explanation": _match_any(rules.explanation, text),
        "proc_verification": _match_any(rules.verification, text),
        "proc_prompt_improve": _match_any(rules.prompt_improve, text),
        "proc_regulation": _match_any(rules.regulation, text),
    }
    return out


def run_regex_classifier(df: pd.DataFrame, text_col: str = TEXT_COL) -> pd.DataFrame:
    df = df.copy()
    preds = df[text_col].apply(lambda t: regex_classify_row(t))
    pred_df = pd.DataFrame(list(preds))
    return pd.concat([df.reset_index(drop=True), pred_df.reset_index(drop=True)], axis=1)




In [ ]:
# ============================================================
# 4) LLM-based classifier (pluggable interface)
# ============================================================

LLM_SYSTEM_PROMPT = """
You are an expert annotator coding student utterances in AI-assisted programming for epistemic aims and processes.

Code the utterance using ONLY the following labels:

AIMS:
- aim_relevance: "on-topic" or "off-topic"
- aim_conceptual: 0 or 1
- aim_procedural: 0 or 1

PROCESSES (each 0 or 1):
- proc_outsourcing
- proc_explanation
- proc_verification
- proc_prompt_improve
- proc_regulation

Definitions (brief):
- outsourcing: asks for complete answer/code with minimal cognitive engagement
- explanation: asks for why/how/meaning
- verification: asks to check correctness/refine/edge cases
- prompt_improve: asks to rephrase/simplify/step-by-step
- regulation: expresses uncertainty + seeks next-step actions, more evidence/examples, or how to avoid mistakes

Return ONLY valid JSON with these keys.
""".strip()


def build_llm_user_prompt(utterance: str) -> str:
    return f'Student utterance:\n"""\n{utterance}\n"""'


# --- Implement ONE of the following callers. Keep interface consistent. ---

def llm_call_stub(system_prompt: str, user_prompt: str) -> str:
    """
    STUB: replace with your actual LLM call.
    Return the assistant text (JSON string).
    """
    raise NotImplementedError(
        "Implement llm_call_stub() with your provider (OpenAI/Anthropic/local). "
        "It should return a JSON string."
    )


def safe_parse_llm_json(text: str) -> Dict[str, object]:
    """
    Make LLM output robust:
      - Extract first JSON object if extra text appears
      - Validate keys
    """
    if not isinstance(text, str):
        raise ValueError("LLM output is not a string")

    # Try direct parse
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        # Fallback: find first {...}
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            raise
        obj = json.loads(m.group(0))

    required_keys = [
        "aim_relevance", "aim_conceptual", "aim_procedural",
        "proc_outsourcing", "proc_explanation", "proc_verification",
        "proc_prompt_improve", "proc_regulation"
    ]
    for k in required_keys:
        if k not in obj:
            raise ValueError(f"Missing key in LLM output: {k}")

    # Normalize types
    obj["aim_relevance"] = str(obj["aim_relevance"]).strip().lower()
    if obj["aim_relevance"] not in ["on-topic", "off-topic"]:
        # allow minor variants
        obj["aim_relevance"] = "on-topic" if "on" in obj["aim_relevance"] else "off-topic"

    for k in required_keys[1:]:
        obj[k] = 1 if str(obj[k]).strip() in ["1", "true", "True"] else 0

    return obj


def run_llm_classifier(
    df: pd.DataFrame,
    text_col: str = TEXT_COL,
    max_rows: Optional[int] = None
) -> pd.DataFrame:
    """
    Runs LLM classification row by row.
    Tip: start with a small max_rows for cost/time control.
    """
    df = df.copy()
    if max_rows is not None:
        df = df.head(max_rows).copy()

    outputs = []
    for _, row in df.iterrows():
        utt = row.get(text_col, "")
        try:
            raw = llm_call_stub(LLM_SYSTEM_PROMPT, build_llm_user_prompt(utt))
            parsed = safe_parse_llm_json(raw)
        except Exception as e:
            parsed = {
                "aim_relevance": None,
                "aim_conceptual": None,
                "aim_procedural": None,
                "proc_outsourcing": None,
                "proc_explanation": None,
                "proc_verification": None,
                "proc_prompt_improve": None,
                "proc_regulation": None,
                "llm_error": str(e)
            }
        outputs.append(parsed)

    out_df = pd.DataFrame(outputs)
    # ensure consistent columns
    if "llm_error" not in out_df.columns:
        out_df["llm_error"] = None

    return pd.concat([df.reset_index(drop=True), out_df.reset_index(drop=True)], axis=1)


# ============================================================
# 5) Pseudo-success detector (based on your 8 indicators only)
# ============================================================

def detect_pseudo_success(df: pd.DataFrame, source_prefix: str = "") -> pd.Series:
    """
    Minimal operationalization using your indicators:
      pseudo_success = outsourcing==1 AND verification==0 AND regulation==0
    source_prefix lets you run on different columns:
      e.g., source_prefix="regex_" if your columns are regex_proc_outsourcing, etc.
    """
    o = df[f"{source_prefix}proc_outsourcing"]
    v = df[f"{source_prefix}proc_verification"]
    r = df[f"{source_prefix}proc_regulation"]
    return (o == 1) & (v == 0) & (r == 0)




In [ ]:
# ============================================================
# 6) Compare methods (manual vs regex vs llm)
# ============================================================

LABEL_COLS_BINARY = [
    "aim_conceptual", "aim_procedural",
    "proc_outsourcing", "proc_explanation", "proc_verification",
    "proc_prompt_improve", "proc_regulation"
]

def compare_binary_labels(
    df: pd.DataFrame,
    human_prefix: str = "human_",
    pred_prefix: str = "regex_",
    label_cols: List[str] = LABEL_COLS_BINARY
) -> pd.DataFrame:
    """
    Expects columns like:
      human_proc_outsourcing, regex_proc_outsourcing
    Computes Cohen's kappa per label + basic classification report.
    """
    rows = []
    for col in label_cols:
        h = df[f"{human_prefix}{col}"].dropna()
        p = df.loc[h.index, f"{pred_prefix}{col}"].dropna()
        common = h.index.intersection(p.index)
        if len(common) < 5:
            rows.append({"label": col, "n": len(common), "kappa": None})
            continue
        kappa = cohen_kappa_score(df.loc[common, f"{human_prefix}{col}"], df.loc[common, f"{pred_prefix}{col}"])
        rows.append({"label": col, "n": len(common), "kappa": float(kappa)})
    return pd.DataFrame(rows)


def error_analysis_table(
    df: pd.DataFrame,
    label: str,
    human_prefix: str = "human_",
    pred_prefix: str = "regex_",
    text_col: str = TEXT_COL,
    n: int = 50
) -> pd.DataFrame:
    """
    Show disagreements for a given label so you can refine regex rules or LLM prompt.
    """
    hcol = f"{human_prefix}{label}"
    pcol = f"{pred_prefix}{label}"
    sub = df[[ID_COL, text_col, hcol, pcol]].dropna()
    sub = sub[sub[hcol] != sub[pcol]].copy()
    return sub.head(n)


def example_workflow(input_path: str):
    # Load
    df = load_data(input_path)
    df = ensure_id_column(df)

    # Ensure your TEXT_COL exists; if not, map it here.
    if TEXT_COL not in df.columns:
        raise ValueError(f"Missing {TEXT_COL}. Please map your student utterance column to TEXT_COL.")

    # 1) Manual coding sheet
    sample_df = sample_for_manual_coding(df, n=200, seed=42)
    ann = make_annotation_sheet(sample_df)
    ann.to_csv("manual_annotation_sheet.csv", index=False)
    print("Saved manual_annotation_sheet.csv")

    # 2) Regex predictions on full df
    df_regex = run_regex_classifier(df, text_col=TEXT_COL)

    # Rename regex columns with prefix so we can compare later
    rename_map = {c: f"regex_{c}" for c in df_regex.columns if c.startswith("aim_") or c.startswith("proc_")}
    df_regex = df_regex.rename(columns=rename_map)

    # Add pseudo-success flag (regex)
    df_regex["regex_pseudo_success"] = detect_pseudo_success(df_regex, source_prefix="regex_").astype(int)

    df_regex.to_csv("regex_predictions.csv", index=False)
    print("Saved regex_predictions.csv")

    # 3) LLM predictions (start small!)
    # df_llm = run_llm_classifier(df, text_col=TEXT_COL, max_rows=50)
    # ... then similarly prefix columns with llm_
    # (You’ll do this after implementing llm_call_stub.)

    return df_regex

